# 📳 Project 5.2 — Vibration Spike Detector
**DSA for Mechatronics · Week 05 — Stacks & Queues**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
An accelerometer on a rotating spindle produces a stream of vibration readings (mm/s²).
The condition monitoring system must detect **spikes** — readings that are the maximum
in a sliding window of W samples — in real time.

A **monotonic deque** solves this in O(n) total time:
the deque always holds indices in decreasing order of value,
so the front is always the current window maximum.
Your dataset is a unique vibration recording seeded from your ID.


## Step 1 — Generate your vibration signal

In [ ]:
import math
from collections import deque

# ── Personalised parameters ─────────────────────────────────────────
N_SAMPLES    = random.randint(80, 140)
BASE_AMP     = round(random.uniform(1.5, 4.0), 2)
NOISE_STD    = round(random.uniform(0.3, 0.8), 2)
N_SPIKES     = random.randint(3, 7)
SPIKE_AMP    = round(random.uniform(8.0, 18.0), 2)
WINDOW_SIZE  = random.randint(7, 15)
ALERT_MULT   = round(random.uniform(2.5, 4.0), 1)   # alert if max > ALERT_MULT * BASE_AMP

# Build signal: sinusoidal base + gaussian noise + random spikes
signal = []
spike_positions = sorted(random.sample(range(5, N_SAMPLES - 5), N_SPIKES))
for i in range(N_SAMPLES):
    val = BASE_AMP * math.sin(2 * math.pi * i / 20) + random.gauss(0, NOISE_STD)
    if i in spike_positions:
        val += SPIKE_AMP * random.uniform(0.8, 1.2)
    signal.append(round(val, 3))

ALERT_THRESHOLD = round(ALERT_MULT * BASE_AMP, 2)

print(f"Vibration signal parameters:")
print(f"  Samples          : {N_SAMPLES}")
print(f"  Base amplitude   : ±{BASE_AMP} mm/s²")
print(f"  Noise std        : {NOISE_STD} mm/s²")
print(f"  Spike count      : {N_SPIKES}  at positions {spike_positions}")
print(f"  Spike amplitude  : ~{SPIKE_AMP} mm/s²")
print(f"  Window size (W)  : {WINDOW_SIZE} samples")
print(f"  Alert threshold  : {ALERT_THRESHOLD} mm/s²  ({ALERT_MULT}× base)")
print()
print(f"  Signal preview (first 20 samples):")
for i in range(min(20, N_SAMPLES)):
    bar_val = max(0, signal[i])
    marker = " ◄ SPIKE" if i in spike_positions else ""
    print(f"  [{i:3d}] {signal[i]:8.3f} mm/s²{marker}")


## Step 2 — Monotonic deque: sliding window maximum in O(n)

In [ ]:
def sliding_window_max(signal, w):
    """
    Compute the maximum value in every window of size w using a monotonic deque.

    The deque stores INDICES in decreasing order of signal value:
      - Before adding index i: pop from the BACK any index j where signal[j] <= signal[i]
        (those values can never be the maximum while i is in the window).
      - Pop from the FRONT when that index has left the window (index <= i - w).
      - The FRONT of the deque is always the index of the current window maximum.

    Time: O(n) — each index is pushed and popped at most once.
    Space: O(w) — deque holds at most w indices.
    """
    dq  = deque()   # holds indices, front = max
    maxs = []

    for i, val in enumerate(signal):
        # remove indices outside the window
        while dq and dq[0] <= i - w:
            dq.popleft()
        # remove indices from back whose values are <= current (they can't be max)
        while dq and signal[dq[-1]] <= val:
            dq.pop()
        dq.append(i)
        # window is full once i >= w-1
        if i >= w - 1:
            maxs.append((i - w + 1, i, signal[dq[0]]))   # (win_start, win_end, max_val)

    return maxs

window_maxs = sliding_window_max(signal, WINDOW_SIZE)

print(f"Sliding window maximum (W={WINDOW_SIZE}):")
print(f"  Total windows computed: {len(window_maxs)}")
print()
print(f"  {'Window':>10}  {'Max value':>12}  Alert")
print(f"  {'─'*10}  {'─'*12}  {'─'*10}")
n_alerts = 0
alert_windows = []
for start, end, mx in window_maxs:
    alert = mx >= ALERT_THRESHOLD
    if alert:
        n_alerts += 1
        alert_windows.append((start, end, mx))
    flag = "⚠ ALERT" if alert else ""
    if start % 5 == 0 or alert:   # print every 5th + all alerts
        print(f"  [{start:3d}–{end:3d}]  {mx:12.3f}  {flag}")
print(f"\n  Alert windows : {n_alerts}")


## Step 3 — Circular buffer: fixed-size FIFO for real-time streaming

In [ ]:
class CircularBuffer:
    """
    Fixed-size FIFO buffer using a circular array.
    When full, the oldest sample is overwritten.
    All operations O(1): enqueue, dequeue, peek.
    """
    def __init__(self, capacity):
        self._buf  = [None] * capacity
        self._cap  = capacity
        self._head = 0      # next write position
        self._size = 0

    def enqueue(self, val):
        self._buf[self._head] = val
        self._head = (self._head + 1) % self._cap
        if self._size < self._cap:
            self._size += 1

    def snapshot(self):
        """Return current buffer contents in order (oldest first)."""
        if self._size < self._cap:
            start = 0
        else:
            start = self._head   # oldest is just after head when full
        return [self._buf[(start + i) % self._cap] for i in range(self._size)]

    @property
    def is_full(self): return self._size == self._cap
    @property
    def size(self): return self._size

# Stream the signal through a circular buffer of size WINDOW_SIZE
BUF_SIZE = WINDOW_SIZE
cbuf     = CircularBuffer(BUF_SIZE)
buf_maxs = []

for i, val in enumerate(signal):
    cbuf.enqueue(val)
    if cbuf.is_full:
        buf_maxs.append(max(cbuf.snapshot()))

print(f"Circular buffer cross-check (capacity = {BUF_SIZE}):")
print(f"  Windows from monotonic deque : {len(window_maxs)}")
print(f"  Windows from circular buffer : {len(buf_maxs)}")

# Verify both methods agree
deque_vals = [mx for _, _, mx in window_maxs]
matches    = sum(1 for a, b in zip(deque_vals, buf_maxs) if abs(a - b) < 1e-9)
print(f"  Matching results             : {matches}/{min(len(deque_vals), len(buf_maxs))} ✅")
print()
print(f"  Final buffer snapshot (last {BUF_SIZE} samples): {cbuf.snapshot()}")


## Step 4 — Alert statistics

In [ ]:
overall_max = max(signal)
overall_min = min(signal)
overall_avg = round(sum(signal) / len(signal), 3)
spike_values = [signal[p] for p in spike_positions]

print("Signal statistics:")
print(f"  Total samples   : {N_SAMPLES}")
print(f"  Min value       : {overall_min:.3f} mm/s²")
print(f"  Max value       : {overall_max:.3f} mm/s²  (at sample {signal.index(overall_max)})")
print(f"  Average value   : {overall_avg:.3f} mm/s²")
print(f"  Alert threshold : {ALERT_THRESHOLD} mm/s²")
print(f"  Alert windows   : {n_alerts} / {len(window_maxs)}")
print()
if alert_windows:
    print("Alert window details:")
    for start, end, mx in alert_windows:
        print(f"  [{start:3d}–{end:3d}]  peak = {mx:.3f} mm/s²  "
              f"({mx/ALERT_THRESHOLD*100:.0f}% of threshold)")


## 📋 Final Report

In [ ]:
W = 56
print("╔" + "═"*W + "╗")
print("║" + " VIBRATION SPIKE DETECTOR — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<26}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<26}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  SIGNAL PARAMETERS" + " "*(W-19) + "║")
print(f"║  {'Samples':<26}: {N_SAMPLES:<26}║")
print(f"║  {'Base amplitude':<26}: ±{BASE_AMP} mm/s²{'':<20}║")
print(f"║  {'Injected spikes':<26}: {N_SPIKES}  at {spike_positions}{'':<5}║"[:W+2] + "║")
print(f"║  {'Window size':<26}: {WINDOW_SIZE} samples{'':<20}║")
print(f"║  {'Alert threshold':<26}: {ALERT_THRESHOLD} mm/s²  ({ALERT_MULT}× base){'':<10}║")
print("╠" + "═"*W + "╣")
print("║  DETECTION RESULTS" + " "*(W-19) + "║")
print(f"║  {'Total windows':<26}: {len(window_maxs):<26}║")
print(f"║  {'Alert windows':<26}: {n_alerts:<26}║")
print(f"║  {'Alert rate':<26}: {n_alerts/len(window_maxs)*100:.1f}%{'':<23}║")
print(f"║  {'Signal max value':<26}: {overall_max:.3f} mm/s²{'':<18}║")
print(f"║  {'Deque/buffer agreement':<26}: {matches}/{min(len(deque_vals),len(buf_maxs))} windows match{'':<10}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which stack or queue concept did you find most important, and why?**
Pick one technique from the notebook and explain in your own words what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
